<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 RAG - Retrieval Augmented Generation
## From Chunking to Grounded Generation with Modern Retrieval Techniques

By the end, you will understand:
- What RAG is and why it exists
- How embeddings, chunking, and retrieval work
- Modern retrieval techniques (hybrid search, re-ranking, query expansion)
- How grounding prevents hallucination
- How to evaluate RAG quality

---

## 📦 Install Dependencies
Run this once before starting.

In [ ]:
!pip uninstall -y \
  google-adk \
  opentelemetry-exporter-gcp-logging \
  opentelemetry-exporter-otlp-proto-http

In [ ]:
!pip install -q \
  chromadb==1.0.20 \
  sentence-transformers \
  anthropic \
  rank-bm25 \
  scikit-learn

---
## Section 1: What Is RAG? Why Does It Exist?

### The Hallucination Problem

Large Language Models (LLMs) are trained on data with a knowledge cutoff — they have no
knowledge of events after training.

More critically, LLMs **hallucinate** — they generate confident, fluent, plausible-sounding
answers that are **completely false**.

```
User: "What is your company's refund policy on digital products?"
LLM (without RAG):
  "Digital products typically have a 30-day refund window,
   though some may have restrictions on redownloads..."
   ← Completely made up! The company may have a completely
     different policy, or no digital products at all.
```

### Why This Happens

LLMs learn patterns from training data. They excel at recognising patterns and generating
text that *looks* like a good answer — even if it's factually wrong.

For **proprietary documents** (company policies, internal research, recent news), the LLM
has **zero specific knowledge** — so it guesses based on patterns it learned elsewhere.

### RAG: The Solution

**Retrieval-Augmented Generation (RAG)** was introduced by Lewis et al. (Facebook AI, 2020)
and remains the dominant approach for document-grounded AI.

RAG solves hallucination by:
1. **Retrieving** the relevant parts of your documents first
2. **Augmenting** the LLM's context with those retrieved parts
3. **Generating** an answer grounded in the documents — not the LLM's training weights

> 🔑 **Key insight:** RAG does NOT fine-tune the LLM. It changes what *context* the LLM
> sees at inference time.

```
User: "What is your company's refund policy on digital products?"
RAG Pipeline:
  1. Search your documents for "refund digital"
  2. Find: "Digital products are non-refundable after purchase."
  3. Pass this to the LLM as context
LLM (with RAG):
  "According to your policy, digital products are
   non-refundable after purchase."
   ← Grounded, factually correct, traceable to source!
```

### The Five-Stage Pipeline

```
YOUR DOCUMENTS (PDFs, text files, web pages)
      │
      ├─ STAGE 1: CHUNKING
      │  Split docs into small, meaningful pieces
      │  "The refund policy covers..." → chunks of 200–500 tokens
      │
      ├─ STAGE 2: EMBEDDING
      │  Convert each chunk to a vector (384 or 768 numbers)
      │  "refund policy" → [0.12, -0.45, 0.89, ..., 0.23]
      │
      ├─ STAGE 3: STORAGE
      │  Store chunks + embeddings + metadata in a vector database
      │
      └─ INDEX READY (built once, queried many times)
          │
          ▼
      USER QUERY: "Are digital products refundable?"
          │
          ├─ STAGE 4: RETRIEVAL
          │  Embed the query (same model!)
          │  Find top-K most similar chunks in database
          │
          ├─ STAGE 5: GENERATION
          │  Pass chunks to LLM as context
          │  LLM reads context + generates grounded answer
          │
          ▼
      ANSWER: "According to your policy, digital products are
               non-refundable." ← Factually correct, traceable to source
```

### When RAG Excels vs. Struggles

| Scenario | Without RAG | With RAG |
|---|---|---|
| **Proprietary docs** (company policies) | LLM guesses ❌ | Retrieved from docs ✅ |
| **Real-time data** (news, market data) | Outdated ❌ | Fresh from document ✅ |
| **Accuracy requirement** (legal, medical) | Risky hallucinations ❌ | Grounded, verifiable ✅ |
| **Citations needed** | No source ❌ | Source chunk provided ✅ |
| **Multi-hop reasoning** across many docs | ❌ Hard | ❌ Still hard |
| **Complex logical inference** | ❌ Hard | ❌ Still hard |
| **Contradictory sources** | ❌ Guesses | ❌ May pick wrong one |


---
## Section 2: Embeddings 101 (Recap + Deep Dive)

### What Is an Embedding?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of text.

**Intuition:**
- Word `"dog"` → embedding `[0.1, 0.8, -0.2, 0.5, ...]`
- Word `"puppy"` → embedding `[0.12, 0.79, -0.19, 0.52, ...]` ← Very similar!
- Word `"cat"` → embedding `[0.15, 0.5, 0.3, 0.6, ...]` ← Somewhat similar
- Word `"pizza"` → embedding `[0.9, 0.2, -0.7, 0.1, ...]` ← Very different!

**Why similar vectors?** The embedding model was trained on millions of sentences where
`"dog"` and `"puppy"` appear in similar contexts. The model learned to place semantically
similar words close together in vector space.

### How Embeddings Are Computed

```
Input text: "Digital products are non-refundable"
     │
     ▼
[Tokenization] Break into tokens:
  ["digital", "products", "are", "non", "-", "refundable"]
     │
     ▼
[Embedding Model - Transformer Neural Network]
  Reads all tokens, understands context, outputs one vector
     │
     ▼
Output: [0.23, -0.45, 0.67, ..., -0.12]  (384 or 768 dimensions)
```

### Distance Metrics: How to Measure Similarity

**Cosine Similarity (Standard for embeddings)**

```
Formula: cos(θ) = (A · B) / (||A|| × ||B||)

Score range: -1.0 to 1.0
  1.0  → identical meaning
  0.5  → related
  0.0  → unrelated
 -1.0  → opposite meaning (rare for text)

Why use it: measures direction only, not magnitude.
            "embedding" and "embeddings are very useful" point the
            same way even though one is much longer.
```


### Embedding Models: Which to Choose?

| Model | Dimensions | Speed | Quality | Best For |
|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Very fast | ~80% | Prototyping, real-time |
| `all-mpnet-base-v2` | 768 | Medium | ~95% | Most production RAG |
| `bge-large-en-v1.5` | 1024 | Slower | ~99% | Legal, medical, high accuracy |
| `multilingual-e5-large` | 1024 | Slower | Excellent | Multilingual documents |

> ⚠️ **Critical:** You MUST use the **same embedding model** at index time and query time.
> Mixing models causes massive precision loss!

### Decision Tree for Model Selection

```
├─ Real-time latency critical?
│  └─ Yes → all-MiniLM-L6-v2 (384 dims, ~5ms per query)
│
├─ Domain-specific (medical, legal, science)?
│  └─ Yes → fine-tuned model from HuggingFace Model Hub
│
├─ Multilingual?
│  └─ Yes → multilingual-e5-large
│
└─ Default → all-mpnet-base-v2 (768 dims)
   Good quality, reasonable speed, well-studied
```


---
## Section 3: Chunking Strategies (Complete Guide)

### Why Chunking Matters

Imagine a 50-page customer handbook (50,000 tokens). If you embed the entire document as
**one vector**, you lose all granularity:

**Problem 1: Loss of Granularity**
```
Query: "What is the warranty on batteries?"
→ The embedding averages over ALL 50 pages.
→ The similarity score is diluted and low, even though page 12 is relevant.
```

**Problem 2: Context Overflow**
```
One chunk = one 50-page document = 50,000 tokens of LLM context.
→ LLM attention gets diluted and it loses focus.
```

**Solution:** Split the document into chunks of 200–600 tokens each.
Each chunk gets its own vector — retrieval finds *exactly* the right section.

### Key Parameters

| Parameter | Typical Value | Effect |
|---|---|---|
| `chunk_size` | 200–600 tokens | Smaller = more focused; larger = more context |
| `overlap` | 20–50 tokens | Prevents losing context at boundaries |

> 🔑 1 token ≈ 4 characters (rough approximation for English text)

### Strategy 1: Fixed-Size Chunking

- Splits every N characters (fastest, simplest)
- **Pros:** Simple, deterministic
- **Cons:** May cut mid-sentence, loses semantic coherence

### Strategy 2: Sentence-Aware Chunking

- Accumulates sentences until chunk is full
- **Pros:** Preserves sentence boundaries, more meaningful embeddings
- **Cons:** Variable chunk sizes

### Strategy 3: Recursive Chunking (Recommended)

- Tries paragraph → sentence → word boundaries in order
- Industry-standard (used by LangChain's `RecursiveCharacterTextSplitter`)
- **Pros:** Balances structure and simplicity, handles mixed formats
- **Cons:** Slightly more complex

### Strategy 4: Semantic Chunking (Advanced)

- Embeds each sentence, splits where similarity drops (topic change detected)
- **Pros:** Chunks align with actual topic boundaries
- **Cons:** Slow (requires embedding every sentence), requires threshold tuning

### Overlapping Chunks: Preventing Context Loss

```
Without overlap (problem):
  Chunk 0: "...The refund policy covers all items purchased within 30 days."
  Chunk 1: "Returns must be in original packaging. Shipping costs..."
  → A query about "original packaging" spans the boundary!

With overlap (solution):
  Chunk 0: "...The refund policy covers all items purchased within 30 days."
  Chunk 1: "...The refund policy covers all items purchased within 30 days.
            Returns must be in original packaging..."
  → Overlap = 20–50 tokens. Now the boundary context is preserved!
```

### Chunk Size by Document Type

| Document Type | Chunk Size (tokens) | Reason |
|---|---|---|
| Technical (API docs) | 200–300 | Dense info, needs focus |
| News articles | 300–500 | Paragraph-level coherence |
| Books, long-form | 400–600 | Complex ideas need context |
| Chat transcripts | 100–200 | Short exchanges |
| Code snippets | 100–250 | Functions are self-contained |


In [ ]:
# ── Section 3 Demo: Implementing all chunking strategies ─────────────

import re  # Regular expressions for sentence splitting
from sklearn.metrics.pairwise import cosine_similarity  # Cosine similarity function
import numpy as np  # Numerical array operations
from sentence_transformers import SentenceTransformer  # Embedding model library

print("Loading embedding model...")  # Status message
model = SentenceTransformer("all-MiniLM-L6-v2")  # Load lightweight model for the demo
print("✅ Model loaded!")  # Confirm load
print()  # Blank line


# Sample document for all demos
SAMPLE_DOCUMENT = """Refund Policy

Digital Products:
Digital products are non-refundable after purchase and download.
This includes software, ebooks, and online courses.
Exceptions may apply in cases of technical failure on our platform.

Physical Products:
Physical goods can be returned within 30 days of purchase.
Items must be unused and in original packaging.
Proof of purchase is required for all returns.

Shipping:
Shipping costs are non-refundable in all circumstances.
Return shipping is free for defective items only.
Standard return shipping is at the customer's expense."""  # Multi-section document


# ─────────────────────────────────────────────────────────────────────
# Strategy 1: Fixed-Size Chunking
# ─────────────────────────────────────────────────────────────────────
def fixed_size_chunking(text, chunk_size=300, overlap=0):
    """
    Split text into fixed-size chunks by character count.

    Args:
        text (str): Full document text
        chunk_size (int): Tokens per chunk (1 token ≈ 4 chars)
        overlap (int): Overlap in tokens between consecutive chunks

    Returns:
        list: List of chunk strings
    """
    char_size = chunk_size * 4  # Convert tokens to characters
    char_overlap = overlap * 4  # Convert overlap tokens to characters
    chunks = []  # Accumulate chunks here
    start = 0  # Start position in text

    while start < len(text):  # Loop until end of document
        end = start + char_size  # Compute end of this chunk
        chunk = text[start:end]  # Slice the chunk
        chunks.append(chunk)  # Add chunk to list
        start = start + char_size - char_overlap  # Advance, leaving overlap

    return chunks  # Return all chunks


# ─────────────────────────────────────────────────────────────────────
# Strategy 2: Sentence-Aware Chunking
# ─────────────────────────────────────────────────────────────────────
def sentence_aware_chunking(text, chunk_size_tokens=300):
    """
    Accumulate sentences into chunks up to chunk_size_tokens.

    Args:
        text (str): Full document
        chunk_size_tokens (int): Target tokens per chunk

    Returns:
        list: List of chunk strings
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)  # Split on sentence boundaries
    chunks = []  # Accumulate chunks here
    current_chunk = ""  # Build current chunk

    for sentence in sentences:  # Process each sentence
        sentence_tokens = len(sentence) / 4  # Rough token estimate
        current_tokens = len(current_chunk) / 4  # Current chunk size

        if current_tokens + sentence_tokens <= chunk_size_tokens:  # Still fits
            current_chunk = current_chunk + " " + sentence  # Append sentence
        else:  # Chunk is full
            if current_chunk.strip():  # Skip empty chunks
                chunks.append(current_chunk.strip())  # Save current chunk
            current_chunk = sentence  # Start new chunk with this sentence

    if current_chunk.strip():  # Handle last chunk
        chunks.append(current_chunk.strip())  # Add last chunk

    return chunks  # Return all chunks


# ─────────────────────────────────────────────────────────────────────
# Strategy 3: Recursive Chunking
# ─────────────────────────────────────────────────────────────────────
def recursive_chunking(text, chunk_size_tokens=300,
                       separators=None):
    """
    Split on successively smaller boundaries: paragraph → sentence → word.

    Args:
        text (str): Full document
        chunk_size_tokens (int): Target tokens per chunk
        separators (list): Ordered list of separators to try

    Returns:
        list: List of chunk strings
    """
    if separators is None:  # Use default hierarchy
        separators = ["\n\n", "\n", ". ", " "]  # paragraph, line, sentence, word

    char_size = chunk_size_tokens * 4  # Convert tokens to characters
    good_splits = []  # Splits found with best separator
    best_sep = separators[-1]  # Default to smallest separator

    for sep in separators:  # Try separators from largest to smallest
        if sep in text:  # This separator exists in text
            good_splits = text.split(sep)  # Split on it
            best_sep = sep  # Remember which separator worked
            break  # Stop — use the largest boundary that works

    if not good_splits:  # No separator found
        good_splits = [text[i:i + char_size]  # Fall back to fixed-size split
                       for i in range(0, len(text), char_size)]

    chunks = []  # Accumulate final chunks
    current_chunk = ""  # Build current chunk

    for split in good_splits:  # Merge splits into target-size chunks
        if current_chunk:  # Append separator between parts
            current_chunk = current_chunk + best_sep + split  # Reconnect
        else:  # First split starts the chunk
            current_chunk = split  # Initialise chunk

        if len(current_chunk) >= char_size:  # Chunk is full
            chunks.append(current_chunk)  # Save it
            current_chunk = ""  # Reset

    if current_chunk:  # Handle remaining text
        chunks.append(current_chunk)  # Add last chunk

    return chunks  # Return all chunks


# ─────────────────────────────────────────────────────────────────────
# Strategy 4: Semantic Chunking
# ─────────────────────────────────────────────────────────────────────
def semantic_chunking(text, embed_model, similarity_threshold=0.5):
    """
    Split where embedding similarity drops between consecutive sentences.

    Args:
        text (str): Full document
        embed_model: SentenceTransformer model
        similarity_threshold (float): Below this = new chunk

    Returns:
        list: Topic-coherent chunks
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)  # Split into sentences first
    sentences = [s.strip() for s in sentences if s.strip()]  # Remove empty strings

    if len(sentences) <= 1:  # Single sentence — return as-is
        return sentences

    embeddings = embed_model.encode(sentences)  # Embed all sentences
    chunks = []  # Accumulate chunks
    current_chunk = sentences[0]  # Start with first sentence

    for i in range(1, len(sentences)):  # Compare consecutive sentences
        prev_emb = embeddings[i - 1].reshape(1, -1)  # Previous sentence embedding
        curr_emb = embeddings[i].reshape(1, -1)  # Current sentence embedding
        sim = cosine_similarity(prev_emb, curr_emb)[0][0]  # Compute similarity

        if sim < similarity_threshold:  # Topic boundary detected
            chunks.append(current_chunk)  # Save current chunk
            current_chunk = sentences[i]  # Start new chunk
        else:  # Same topic — continue building chunk
            current_chunk = current_chunk + " " + sentences[i]  # Append

    chunks.append(current_chunk)  # Add final chunk

    return chunks  # Return topic-coherent chunks


# ─────────────────────────────────────────────────────────────────────
# Overlapping Chunks
# ─────────────────────────────────────────────────────────────────────
def overlapping_chunks(text, chunk_size_tokens=300, overlap_tokens=50):
    """
    Fixed-size chunks with overlap to prevent boundary context loss.

    Args:
        text (str): Full document
        chunk_size_tokens (int): Tokens per chunk
        overlap_tokens (int): Overlapping tokens between consecutive chunks

    Returns:
        list: List of overlapping chunk strings
    """
    char_size = chunk_size_tokens * 4  # Convert to characters
    char_overlap = overlap_tokens * 4  # Convert overlap to characters
    chunks = []  # Accumulate chunks
    start = 0  # Start position

    while start < len(text):  # Loop until end of document
        end = min(start + char_size, len(text))  # Don't exceed text length
        chunk = text[start:end]  # Extract chunk
        chunks.append(chunk)  # Add to list
        start = start + char_size - char_overlap  # Advance with overlap

    return chunks  # Return all overlapping chunks


# ─────────────────────────────────────────────────────────────────────
# Run all strategies on the sample document
# ─────────────────────────────────────────────────────────────────────
print("=" * 60)  # Top separator
print("CHUNKING STRATEGIES COMPARISON")  # Header
print("=" * 60)  # Bottom separator
print(f"Document length: {len(SAMPLE_DOCUMENT)} characters "  # Show doc size
      f"(~{len(SAMPLE_DOCUMENT)//4} tokens)")  # Approximate token count
print()  # Blank line

# Strategy 1: Fixed-size
fixed_chunks = fixed_size_chunking(SAMPLE_DOCUMENT, chunk_size=50)  # 50-token demo
print(f"Strategy 1 — Fixed-Size (chunk_size=50 tokens): {len(fixed_chunks)} chunks")  # Count
for i in range(min(2, len(fixed_chunks))):  # Show first 2 chunks
    print(f"  Chunk {i}: [{len(fixed_chunks[i])//4} tokens] {fixed_chunks[i][:60]!r}...")  # Preview
print()  # Blank line

# Strategy 2: Sentence-aware
sentence_chunks = sentence_aware_chunking(SAMPLE_DOCUMENT, chunk_size_tokens=50)  # 50-token demo
print(f"Strategy 2 — Sentence-Aware (chunk_size=50 tokens): {len(sentence_chunks)} chunks")  # Count
for i in range(min(2, len(sentence_chunks))):  # Show first 2 chunks
    print(f"  Chunk {i}: [{len(sentence_chunks[i])//4} tokens] {sentence_chunks[i][:60]!r}...")  # Preview
print()  # Blank line

# Strategy 3: Recursive
recursive_chunks = recursive_chunking(SAMPLE_DOCUMENT, chunk_size_tokens=50)  # 50-token demo
print(f"Strategy 3 — Recursive (chunk_size=50 tokens): {len(recursive_chunks)} chunks")  # Count
for i in range(min(2, len(recursive_chunks))):  # Show first 2 chunks
    print(f"  Chunk {i}: [{len(recursive_chunks[i])//4} tokens] {recursive_chunks[i][:60]!r}...")  # Preview
print()  # Blank line

# Strategy 4: Semantic
semantic_chunks_list = semantic_chunking(SAMPLE_DOCUMENT, model, similarity_threshold=0.4)  # Use loaded model
print(f"Strategy 4 — Semantic (threshold=0.4): {len(semantic_chunks_list)} chunks")  # Count
for i in range(min(2, len(semantic_chunks_list))):  # Show first 2 chunks
    print(f"  Chunk {i}: [{len(semantic_chunks_list[i])//4} tokens] {semantic_chunks_list[i][:60]!r}...")  # Preview
print()  # Blank line

# Strategy 5: Overlapping
overlap_chunks = overlapping_chunks(SAMPLE_DOCUMENT, chunk_size_tokens=50, overlap_tokens=10)  # 10-token overlap
print(f"Overlapping Chunks (chunk=50, overlap=10 tokens): {len(overlap_chunks)} chunks")  # Count
for i in range(min(2, len(overlap_chunks))):  # Show first 2 chunks
    print(f"  Chunk {i}: [{len(overlap_chunks[i])//4} tokens] {overlap_chunks[i][:60]!r}...")  # Preview
print()  # Blank line
print("Observation: Sentence-aware and recursive strategies produce")  # Summary
print("more meaningful chunks than fixed-size splitting.")  # Summary continued

---
## Section 4: Storing Chunks & Metadata

### What Gets Stored Per Chunk

For each chunk, the vector database stores:

```python
chunk_record = {
    "id": "doc_refund_chunk_3",           # Unique identifier
    "document": "Digital products are...",  # Raw chunk text (for generation)
    "embedding": [0.12, -0.45, ...],        # Vector (for similarity search)
    "metadata": {
        "source_file": "refund_policy.pdf",
        "chunk_index": 3,
        "date_added": "2024-06-01",
        "category": "company_policies",
        "language": "english"
    }
}
```

### Metadata for Filtering

ChromaDB (and other vector databases) support pre-filtering before semantic search:

```python
# Only retrieve from recent documents (date filter)
results = collection.query(
    query_texts=["refund policy"],
    n_results=5,
    where={"date_added": {"$gte": "2024-01-01"}}
)

# Multiple filters (AND logic)
results = collection.query(
    query_texts=["refund policy"],
    n_results=5,
    where={
        "$and": [
            {"date_added": {"$gte": "2024-01-01"}},
            {"confidentiality": "public"},
            {"language": "en"}
        ]
    }
)
```

> ChromaDB, Pinecone, and Weaviate all support metadata filtering natively.


In [ ]:
# ── Section 4 Demo: Creating a ChromaDB collection and adding chunks ──
import chromadb  # Vector database client


# ── Set up ChromaDB client ─────────────────────────────────────────────
chroma_client = chromadb.Client()  # In-memory client (no disk persistence)

# ── Create a collection ───────────────────────────────────────────────
collection = chroma_client.create_collection(  # Create named collection
    name="company_policies",  # Collection name
    metadata={"description": "Company policy documents"}  # Collection metadata
)  # End of create_collection

# ── Define sample chunks with rich metadata ────────────────────────────
chunks_data = [  # List of chunk records
    {  # Chunk 1
        "id": "refund_digital_1",  # Unique ID
        "text": "Digital products are non-refundable after purchase and download.",  # Chunk text
        "metadata": {  # Metadata dict
            "source": "refund_policy.pdf",  # Source file
            "chunk_index": 0,  # Position in document
            "category": "refunds",  # Topic category
            "date_added": "2024-06-01",  # Ingestion date
            "language": "en"  # Language
        }  # End metadata
    },  # End chunk 1
    {  # Chunk 2
        "id": "refund_physical_1",  # Unique ID
        "text": "Physical goods can be returned within 30 days of purchase.",  # Chunk text
        "metadata": {  # Metadata dict
            "source": "refund_policy.pdf",  # Source file
            "chunk_index": 1,  # Position in document
            "category": "refunds",  # Topic category
            "date_added": "2024-06-01",  # Ingestion date
            "language": "en"  # Language
        }  # End metadata
    },  # End chunk 2
    {  # Chunk 3
        "id": "shipping_1",  # Unique ID
        "text": "Shipping costs are non-refundable under all circumstances.",  # Chunk text
        "metadata": {  # Metadata dict
            "source": "shipping_policy.pdf",  # Source file
            "chunk_index": 0,  # Position in document
            "category": "shipping",  # Topic category
            "date_added": "2024-06-01",  # Ingestion date
            "language": "en"  # Language
        }  # End metadata
    },  # End chunk 3
    {  # Chunk 4
        "id": "warranty_1",  # Unique ID
        "text": "Electronics have a 1-year manufacturer warranty.",  # Chunk text
        "metadata": {  # Metadata dict
            "source": "warranty_policy.pdf",  # Source file
            "chunk_index": 0,  # Position in document
            "category": "warranty",  # Topic category
            "date_added": "2024-06-01",  # Ingestion date
            "language": "en"  # Language
        }  # End metadata
    },  # End chunk 4
    {  # Chunk 5
        "id": "shipping_2",  # Unique ID
        "text": "Return shipping is free for defective items only.",  # Chunk text
        "metadata": {  # Metadata dict
            "source": "shipping_policy.pdf",  # Source file
            "chunk_index": 1,  # Position in document
            "category": "shipping",  # Topic category
            "date_added": "2024-06-01",  # Ingestion date
            "language": "en"  # Language
        }  # End metadata
    }  # End chunk 5
]  # End chunks list

# ── Extract fields for ChromaDB ────────────────────────────────────────
ids = [chunk["id"] for chunk in chunks_data]  # List of IDs
documents = [chunk["text"] for chunk in chunks_data]  # List of texts
metadatas = [chunk["metadata"] for chunk in chunks_data]  # List of metadata dicts

# ── Embed documents manually and add with embeddings ──────────────────
doc_embeddings = model.encode(documents)  # Compute embeddings for all chunks
doc_embeddings_list = doc_embeddings.tolist()  # Convert numpy arrays to Python lists

# ── Add to ChromaDB collection ─────────────────────────────────────────
collection.add(  # Add all chunks at once
    ids=ids,  # Unique IDs
    documents=documents,  # Raw texts
    embeddings=doc_embeddings_list,  # Pre-computed embeddings
    metadatas=metadatas  # Metadata dicts
)  # End add

# ── Verify ────────────────────────────────────────────────────────────
print(f"✅ Added {len(chunks_data)} chunks to collection")  # Confirm
print(f"Total chunks in collection: {collection.count()}")  # Verify count
print()  # Blank line

# ── Show what's stored ────────────────────────────────────────────────
all_data = collection.get()  # Retrieve all stored records
print("Stored chunks:")  # Label
for i in range(len(all_data["ids"])):  # Loop through all records
    chunk_id = all_data["ids"][i]  # Get ID
    chunk_text = all_data["documents"][i]  # Get text
    chunk_meta = all_data["metadatas"][i]  # Get metadata
    print(f"  ID: {chunk_id}")  # Print ID
    print(f"  Text: {chunk_text[:60]}...")  # Print text preview
    print(f"  Source: {chunk_meta["source"]} | Category: {chunk_meta["category"]}")  # Print metadata
    print()  # Blank line between records

---
## Section 5: Retrieval Fundamentals

### How Retrieval Works (Step-by-Step)

```
1. USER ASKS: "Can I return digital products?"

2. ENCODE QUERY (same model used at index time!):
   query_embedding = model.encode("Can I return digital products?")
   → [0.11, -0.44, 0.90, ..., 0.22]  (384 or 768 dimensions)

3. SEARCH VECTOR DATABASE:
   For each stored chunk embedding:
     similarity = cosine_similarity(query_embedding, chunk_embedding)

   Results (sorted by similarity):
   - "Digital products are non-refundable" : sim=0.89 ✅ (highly relevant)
   - "Physical goods can be returned..."   : sim=0.72 ✓  (somewhat relevant)
   - "Shipping costs are non-refundable"   : sim=0.55    (slightly relevant)

4. RETURN TOP-K (K=3 is the safe default):
   LLM will read these chunks as context.
```

### Choosing K (Number of Retrieved Chunks)

| K | Trade-off |
|---|---|
| K=1 | Focused, but may miss related info |
| **K=3** | **Safe default. Covers main answer + context.** |
| K=5 | Comprehensive but LLM may lose focus |
| K>10 | Too much info, LLM confused, higher cost |


In [ ]:
# ── Section 5 Demo: Basic retrieval ──────────────────────────────────

# ── Query 1: Simple retrieval ──────────────────────────────────────────
query_1 = "Can I return digital products?"  # Test query
query_1_emb = model.encode(query_1).tolist()  # Embed query (must use same model!)

results_1 = collection.query(  # Run similarity search
    query_embeddings=[query_1_emb],  # Embedded query vector
    n_results=3  # Top-K = 3
)  # End query

print("=" * 60)  # Separator
print(f"QUERY: {query_1}")  # Show query
print("=" * 60)  # Separator

retrieved_docs = results_1["documents"][0]  # Retrieved chunk texts
retrieved_meta = results_1["metadatas"][0]  # Retrieved metadata
retrieved_dist = results_1["distances"][0]  # Distances (lower = more similar)

for i in range(len(retrieved_docs)):  # Display each result
    distance = retrieved_dist[i]  # L2 distance from ChromaDB
    similarity = 1 / (1 + distance)  # Convert to 0-1 similarity score
    source = retrieved_meta[i]["source"]  # Source file
    category = retrieved_meta[i]["category"]  # Category
    text = retrieved_docs[i]  # Chunk text
    print(f"Rank {i + 1} | Similarity: {similarity:.4f}")  # Print rank and score
    print(f"  Source: {source} | Category: {category}")  # Print metadata
    print(f"  Text: {text}")  # Print full chunk text
    print()  # Blank line

# ── Query 2: Retrieval with metadata filtering ─────────────────────────
query_2 = "What are the refund rules?"  # Second test query
query_2_emb = model.encode(query_2).tolist()  # Embed query

results_2 = collection.query(  # Filtered query
    query_embeddings=[query_2_emb],  # Embedded query
    n_results=3,  # Top-K = 3
    where={"category": "refunds"}  # Only retrieve refund-category chunks
)  # End filtered query

print("=" * 60)  # Separator
print(f"FILTERED QUERY (category=refunds): {query_2}")  # Show filtered query
print("=" * 60)  # Separator

filtered_docs = results_2["documents"][0]  # Filtered results
filtered_meta = results_2["metadatas"][0]  # Filtered metadata

for i in range(len(filtered_docs)):  # Display each filtered result
    source = filtered_meta[i]["source"]  # Source file
    category = filtered_meta[i]["category"]  # Category
    text = filtered_docs[i]  # Chunk text
    print(f"Rank {i + 1} | Source: {source} | Category: {category}")  # Metadata
    print(f"  Text: {text}")  # Chunk text
    print()  # Blank line

print("Observation: Filtered retrieval returns only refund-category chunks,")  # Summary
print("even if shipping chunks have a slightly higher similarity score.")  # Summary

---
## Section 6: Modern Retrieval Techniques

### 1. Hybrid Retrieval (Dense + Sparse)

**Problem with pure dense retrieval:**
```
Query: "iPhone Model A12 Bionic warranty"
→ Dense similarity: 0.65 (moderate)
→ The embedding may miss the specific product ID "A12 Bionic"
```

**Hybrid combines:**
- **Dense (embedding-based):** semantic understanding
- **Sparse (BM25):** exact keyword matching

```
hybrid_score = α × dense_sim + (1-α) × sparse_score
```

Empirically outperforms either approach alone on standard benchmarks.

**When to use:** E-commerce (SKUs + semantics), technical docs (exact terms), legal/medical.

---

### 2. Re-Ranking (Two-Stage Retrieval)

**Problem:**
```
Dense retrieval gets top-100 fast, but precision is imperfect.
→ Some irrelevant chunks rank high; some relevant chunks rank low.
```

**Solution:**
```
Stage 1: Fast dense retriever → top-100 candidates (milliseconds)
Stage 2: Cross-encoder re-ranker → scores each query+document pair precisely
         (slower, but used by Google, Microsoft Bing, Elasticsearch)
```

Cross-encoders encode the **query and document together** (not independently),
so they understand interaction — improving precision by 5–15%.

**When to use:** When precision matters more than latency (medical, legal).

---

### 3. Query Expansion

**Problem:**
```
Query: "return policy"   →   Document: "refund policy"
Dense similarity: 0.60 (borderline — may not retrieve!)
```

**Solution:** Generate multiple query variants, retrieve independently, merge results.

```
"return policy" expands to:
  1. "return policy"
  2. "refund policy"
  3. "can I return items?"
  4. "how do I get a refund?"
→ Retrieve all 4, merge — now the refund chunk is captured!
```

**When to use:** Domain with many synonyms, when recall > latency.

---

### 4. Multi-Hop Retrieval (Conceptual Overview)

For questions requiring reasoning **across multiple documents**:

```
Q: "How does our refund policy affect quarterly revenue?"

Hop 1: Retrieve refund policy document
Hop 2: LLM generates follow-up question about revenue accounting
Hop 3: Retrieve financial statements
→ LLM synthesises both retrieved contexts into a final answer
```

Used when no single document contains the full answer.

---

### 5. Adaptive Retrieval (Brief Intro)

Decide **whether retrieval is needed at all**:
- If the query can be answered from LLM training data (e.g., "What is 2+2?") → skip retrieval
- If the top-K similarity is below a threshold → skip generation (no relevant docs)

Research has shown this can reduce costs by ~40% with minimal quality loss.


In [ ]:
# ── Section 6 Demo: Hybrid Retrieval + Re-Ranking ─────────────────────

from rank_bm25 import BM25Okapi  # BM25 sparse retrieval library


# ─────────────────────────────────────────────────────────────────────
# Technique 1: Hybrid Retrieval
# ─────────────────────────────────────────────────────────────────────
def hybrid_retrieval(query, chroma_collection, embed_model,
                     all_docs, all_ids, k=3, alpha=0.6):
    """
    Combine dense (embedding) and sparse (BM25) retrieval.

    Args:
        query (str): User question
        chroma_collection: ChromaDB collection
        embed_model: SentenceTransformer model
        all_docs (list): All chunk texts (for BM25)
        all_ids (list): All chunk IDs
        k (int): Final number of results
        alpha (float): Weight for dense score (1-alpha = sparse weight)

    Returns:
        list: Top-k (chunk_text, hybrid_score) tuples
    """
    # ── Dense retrieval ──────────────────────────────────────────────
    query_emb = embed_model.encode(query).tolist()  # Embed query
    dense_results = chroma_collection.query(  # Similarity search
        query_embeddings=[query_emb],  # Query embedding
        n_results=len(all_docs)  # Get all docs to merge
    )  # End dense query

    dense_docs = dense_results["documents"][0]  # Retrieved texts
    dense_dists = dense_results["distances"][0]  # L2 distances
    dense_ids = dense_results["ids"][0]  # Retrieved IDs

    # Convert L2 distances to similarity scores (0-1)
    dense_sims = []  # Accumulate similarity scores
    for dist in dense_dists:  # Loop through distances
        sim = 1 / (1 + dist)  # Invert distance to similarity
        dense_sims.append(sim)  # Add to list

    # Normalise dense scores to 0-1 range
    max_dense = max(dense_sims) if dense_sims else 1.0  # Avoid divide-by-zero
    dense_sims_norm = [s / max_dense for s in dense_sims]  # Normalise

    # ── Sparse retrieval (BM25) ──────────────────────────────────────
    tokenized_docs = [doc.lower().split() for doc in all_docs]  # Tokenise each doc
    bm25 = BM25Okapi(tokenized_docs)  # Build BM25 index
    query_tokens = query.lower().split()  # Tokenise query
    sparse_scores = bm25.get_scores(query_tokens)  # BM25 scores for all docs

    # Normalise sparse scores to 0-1
    max_sparse = max(sparse_scores) if max(sparse_scores) > 0 else 1.0  # Avoid divide-by-zero
    sparse_scores_norm = [s / max_sparse for s in sparse_scores]  # Normalise

    # ── Merge scores ──────────────────────────────────────────────────
    merged = {}  # Map: doc_id → hybrid_score

    for i in range(len(dense_ids)):  # Add dense scores
        doc_id = dense_ids[i]  # Get doc ID
        dense_score = alpha * dense_sims_norm[i]  # Weighted dense score
        merged[doc_id] = dense_score  # Store

    for j in range(len(all_ids)):  # Add sparse scores
        doc_id = all_ids[j]  # Get doc ID
        sparse_score = (1 - alpha) * sparse_scores_norm[j]  # Weighted sparse score
        if doc_id in merged:  # Already has dense score
            merged[doc_id] = merged[doc_id] + sparse_score  # Add sparse
        else:  # Only sparse
            merged[doc_id] = sparse_score  # Sparse only

    # ── Sort and return top-k ─────────────────────────────────────────
    sorted_ids = sorted(merged.items(), key=lambda x: x[1], reverse=True)  # Sort by score
    top_k = sorted_ids[:k]  # Take top-k

    # Build result list: (text, score)
    id_to_text = {}  # Map doc ID to text
    for i in range(len(dense_ids)):  # Build map from dense results
        id_to_text[dense_ids[i]] = dense_docs[i]  # Store
    for j in range(len(all_ids)):  # Also include docs not in dense results
        if all_ids[j] not in id_to_text:  # Not yet mapped
            id_to_text[all_ids[j]] = all_docs[j]  # Add

    results = []  # Accumulate results
    for doc_id, score in top_k:  # Build result tuples
        text = id_to_text.get(doc_id, "")  # Get text
        results.append((text, score, doc_id))  # (text, score, id)

    return results  # Return top-k hybrid results


# ─────────────────────────────────────────────────────────────────────
# Technique 2: Re-Ranking with Cross-Encoder
# ─────────────────────────────────────────────────────────────────────
def retrieval_with_reranking(query, chroma_collection, embed_model,
                             k=3, num_candidates=5):
    """
    Two-stage retrieval: dense retrieval + cross-encoder re-ranking.

    Args:
        query (str): User question
        chroma_collection: ChromaDB collection
        embed_model: SentenceTransformer model
        k (int): Final results to return
        num_candidates (int): Candidates from stage 1 for re-ranking

    Returns:
        list: Top-k (chunk_text, rerank_score) tuples
    """
    from sentence_transformers import CrossEncoder  # Cross-encoder for re-ranking

    # ── Stage 1: Dense retrieval (fast) ──────────────────────────────
    query_emb = embed_model.encode(query).tolist()  # Embed query
    dense_results = chroma_collection.query(  # Similarity search
        query_embeddings=[query_emb],  # Embedded query
        n_results=num_candidates  # Get more candidates for re-ranking
    )  # End dense query

    retrieved_chunks = dense_results["documents"][0]  # Candidate texts
    retrieved_ids = dense_results["ids"][0]  # Candidate IDs

    # ── Stage 2: Re-ranking (slower, more accurate) ───────────────────
    reranker = CrossEncoder(  # Load cross-encoder model
        "cross-encoder/ms-marco-MiniLM-L-6-v2"  # Lightweight cross-encoder
    )  # End CrossEncoder

    # Create (query, chunk) pairs for scoring
    query_chunk_pairs = []  # Accumulate pairs
    for chunk in retrieved_chunks:  # Loop through candidates
        pair = [query, chunk]  # Create pair
        query_chunk_pairs.append(pair)  # Add to list

    rerank_scores = reranker.predict(query_chunk_pairs)  # Score all pairs

    # ── Sort by re-rank scores ────────────────────────────────────────
    scored = list(zip(retrieved_chunks, rerank_scores, retrieved_ids))  # Zip together
    scored.sort(key=lambda x: x[1], reverse=True)  # Sort by score (descending)

    return scored[:k]  # Return top-k after re-ranking


# ─────────────────────────────────────────────────────────────────────
# Technique 3: Query Expansion
# ─────────────────────────────────────────────────────────────────────
def retrieval_with_expansion(query, chroma_collection, embed_model, k=3):
    """
    Expand query to multiple variants, retrieve for each, merge results.

    Args:
        query (str): Original user question
        chroma_collection: ChromaDB collection
        embed_model: SentenceTransformer model
        k (int): Final results to return

    Returns:
        list: Top-k (chunk_text, vote_count) tuples
    """
    # Manually define query variants (in production, use LLM to generate these)
    expanded_queries = [  # Multiple phrasings
        query,  # Original query
        "What is the return policy?",  # Synonym: return → refund
        "Can I get a refund?",  # Paraphrase
        "Are items refundable?",  # Alternative phrasing
    ]  # End expanded queries

    vote_counts = {}  # Map: chunk_text → number of queries that retrieved it

    for expanded_q in expanded_queries:  # Retrieve for each query variant
        q_emb = embed_model.encode(expanded_q).tolist()  # Embed variant
        q_results = chroma_collection.query(  # Retrieve
            query_embeddings=[q_emb],  # Embedded variant
            n_results=k  # Top-K
        )  # End query

        for chunk_text in q_results["documents"][0]:  # Loop through results
            if chunk_text not in vote_counts:  # New chunk
                vote_counts[chunk_text] = 0  # Initialise
            vote_counts[chunk_text] = vote_counts[chunk_text] + 1  # Vote

    # Sort by votes — most-retrieved chunk across variants is most relevant
    sorted_results = sorted(vote_counts.items(),
                            key=lambda x: x[1], reverse=True)  # Sort descending

    return sorted_results[:k]  # Return top-k by vote count


# ─────────────────────────────────────────────────────────────────────
# Run Demos
# ─────────────────────────────────────────────────────────────────────
all_stored = collection.get()  # Retrieve all stored chunks
all_docs_list = all_stored["documents"]  # All chunk texts
all_ids_list = all_stored["ids"]  # All chunk IDs

test_query = "Can I return items and get my money back?"  # Test query

print("=" * 60)  # Separator
print("MODERN RETRIEVAL TECHNIQUES")  # Header
print(f"Query: {test_query}")  # Show query
print("=" * 60)  # Separator
print()  # Blank line

# ── Demo 1: Hybrid Retrieval ──────────────────────────────────────────
print("Technique 1: Hybrid Retrieval (dense + BM25, alpha=0.6)")  # Label
print("-" * 40)  # Separator
hybrid_results = hybrid_retrieval(  # Run hybrid retrieval
    query=test_query,  # User query
    chroma_collection=collection,  # Vector store
    embed_model=model,  # Embedding model
    all_docs=all_docs_list,  # All stored docs
    all_ids=all_ids_list,  # All stored IDs
    k=3,  # Return top-3
    alpha=0.6  # 60% dense, 40% sparse
)  # End hybrid_retrieval
for rank, (text, score, doc_id) in enumerate(hybrid_results):  # Display results
    print(f"  Rank {rank + 1} | Hybrid Score: {score:.4f} | ID: {doc_id}")  # Score
    print(f"    Text: {text}")  # Text
print()  # Blank line

# ── Demo 2: Re-Ranking ────────────────────────────────────────────────
print("Technique 2: Re-Ranking with Cross-Encoder")  # Label
print("-" * 40)  # Separator
print("(Downloading cross-encoder model if not cached — ~50 MB)")  # Note
try:  # Cross-encoder may not be installed
    reranked_results = retrieval_with_reranking(  # Run re-ranking
        query=test_query,  # User query
        chroma_collection=collection,  # Vector store
        embed_model=model,  # Embedding model
        k=3,  # Return top-3
        num_candidates=5  # Candidates for re-ranking
    )  # End retrieval_with_reranking
    for rank, (text, score, doc_id) in enumerate(reranked_results):  # Display
        print(f"  Rank {rank + 1} | Re-rank Score: {score:.4f} | ID: {doc_id}")  # Score
        print(f"    Text: {text}")  # Text
except Exception as e:  # Handle errors gracefully
    print(f"  Re-ranking skipped: {e}")  # Show error
    print("  Install sentence-transformers and cross-encoder model to enable.")  # Tip
print()  # Blank line

# ── Demo 3: Query Expansion ───────────────────────────────────────────
print("Technique 3: Query Expansion (4 variants, voting)")  # Label
print("-" * 40)  # Separator
expansion_results = retrieval_with_expansion(  # Run query expansion
    query=test_query,  # User query
    chroma_collection=collection,  # Vector store
    embed_model=model,  # Embedding model
    k=3  # Return top-3
)  # End retrieval_with_expansion
for rank, (text, votes) in enumerate(expansion_results):  # Display results
    print(f"  Rank {rank + 1} | Votes: {votes}/4")  # Show vote count
    print(f"    Text: {text}")  # Chunk text
print()  # Blank line
print("Observation: Chunks retrieved by more query variants are more likely relevant.")  # Summary

---
## Section 7: Grounding Theory & Implementation

### What Is Grounding?

**Grounding** = constraining the LLM to answer *only* from the retrieved context,
preventing it from using its training knowledge.

```
WITHOUT GROUNDING:
  System prompt: "You are a helpful assistant."
  Retrieved context: (empty or irrelevant)
  User: "What is your refund policy?"

  LLM thinks: "I learned from training that typical policies are 30 days."
  LLM output: "Most companies offer a 30-day refund window..."
              ← Hallucinated!

WITH GROUNDING:
  System prompt: "Answer ONLY from the context below. If not in context,
                  say 'I don't have enough information.'"
  Retrieved context: "Digital products are non-refundable."
  User: "What is your refund policy?"

  LLM output: "According to the context, digital products are non-refundable."
              ← Grounded. Traceable to document.
```

### Grounding System Prompt Design Principles

**Principle 1: Be explicit about the constraint**
```python
# ❌ Too vague
"Use the following information to answer:"

# ✅ Explicit
"Answer the user's question using ONLY the provided context.
Do not use your training knowledge. If the answer is not in the
context, respond: 'I don't have enough information to answer that.'"
```

**Principle 2: Use negative prompting (tell it what NOT to do)**
```python
"Do not:
  - Use information not in the context
  - Make assumptions
  - Guess or extrapolate beyond what's stated
  - Use your general knowledge to fill in gaps"
```

**Principle 3: Give clear examples (few-shot)**
```python
"Example:
  Context: 'Our policy: physical goods can be returned within 30 days.'
  Question: 'Can I return digital products?'
  Good answer: 'The context covers physical goods within 30 days.
               Digital products are not mentioned, so I don't have
               enough information about digital returns.'"
```

### The "I Don't Know" Test (Evaluating Grounding)

A well-grounded system should **correctly say "I don't know"** when the answer
is not in the retrieved context:

```
Query: "What is your employee vacation policy?"
Retrieved chunks: [Only refund policy chunks — no HR policies]

✅ Well-grounded LLM: "I don't have enough information in the
                       provided documents to answer that."

❌ Poorly-grounded LLM: "Typically, companies offer 20 days of
                         vacation per year..."
                         ← Hallucinated from training data!
```


In [ ]:
# ── Section 7 Demo: Grounding implementation (offline, no API key needed)

# ─────────────────────────────────────────────────────────────────────
# Define the grounding system prompt as a module-level constant
# ─────────────────────────────────────────────────────────────────────
GROUNDING_SYSTEM_PROMPT = """  \
You are a helpful assistant answering questions about company policies.

Answer questions using ONLY the context provided below.
Do NOT use your training knowledge or general knowledge.

Rules:
1. Answer ONLY from the provided context.
2. If the answer is not in the context, say:
   "I don't have enough information in the provided documents to answer that."
3. Be concise and reference the relevant parts of the context.

Do NOT:
- Use information not in the context
- Make assumptions
- Guess or extrapolate beyond what is stated
- Use general knowledge to fill gaps

Example:
  Context: "Physical goods can be returned within 30 days."
  Question: "Can I return digital products?"
  Good answer: "The context covers physical goods within 30 days.
               Digital products are not mentioned, so I cannot
               answer that from the provided documents."

Context:
{context}
"""  # End of grounding system prompt


# ─────────────────────────────────────────────────────────────────────
# Helper: Assemble context string from retrieved chunks
# ─────────────────────────────────────────────────────────────────────
def assemble_context(retrieved_chunks):
    """
    Join retrieved chunks into a formatted context string.

    Args:
        retrieved_chunks (list): List of chunk text strings

    Returns:
        str: Formatted context for LLM system prompt
    """
    separator = "\n\n---\n\n"  # Visual separator between chunks
    context = separator.join(retrieved_chunks)  # Join all chunks
    return context  # Return assembled context


def assemble_context_with_sources(retrieved_results):
    """
    Join chunks with source attribution for better traceability.

    Args:
        retrieved_results (list): List of dicts with "document" and "metadata"

    Returns:
        str: Context with source labels
    """
    context_parts = []  # Accumulate formatted parts

    for i in range(len(retrieved_results)):  # Loop through results
        result = retrieved_results[i]  # Current result dict
        chunk_text = result["document"]  # Raw chunk text
        source = result["metadata"].get("source", "Unknown source")  # Source file
        formatted = f"[Source: {source}]\n{chunk_text}"  # Format with source
        context_parts.append(formatted)  # Add to list

    separator = "\n\n---\n\n"  # Separator between chunks
    context = separator.join(context_parts)  # Join all parts
    return context  # Return formatted context


# ─────────────────────────────────────────────────────────────────────
# Helper: Evaluate grounding quality (heuristic, no API needed)
# ─────────────────────────────────────────────────────────────────────
def evaluate_grounding_heuristic(answer, context):
    """
    Heuristic check: is the answer likely grounded in the context?

    Args:
        answer (str): LLM-generated answer
        context (str): Context provided to LLM

    Returns:
        tuple: (is_grounded bool, overlap_ratio float)
    """
    if not context.strip():  # Empty context
        if "don't have" in answer.lower() or "not enough" in answer.lower():  # Admitted ignorance
            return True, 1.0  # Correctly said "I don't know"
        else:
            return False, 0.0  # Should have said "I don't know"

    context_words = set(context.lower().split())  # Words in context
    answer_words = set(answer.lower().split())  # Words in answer
    overlap = context_words & answer_words  # Intersection

    if not answer_words:  # Empty answer
        return False, 0.0  # No answer = not grounded

    overlap_ratio = len(overlap) / len(answer_words)  # Overlap fraction
    is_grounded = overlap_ratio >= 0.3  # 30% overlap threshold

    return is_grounded, overlap_ratio  # Return result


# ─────────────────────────────────────────────────────────────────────
# Demo: Show grounding prompt and context assembly
# ─────────────────────────────────────────────────────────────────────
print("=" * 60)  # Separator
print("GROUNDING DEMO")  # Header
print("=" * 60)  # Separator
print()  # Blank line

# Retrieve chunks for a query
demo_query = "Can I return digital products?"  # Demo query
demo_query_emb = model.encode(demo_query).tolist()  # Embed query
demo_results = collection.query(  # Retrieve
    query_embeddings=[demo_query_emb],  # Embedded query
    n_results=3  # Top-3
)  # End query

demo_chunks = demo_results["documents"][0]  # Retrieved texts
demo_meta = demo_results["metadatas"][0]  # Retrieved metadata

# Assemble context with source attribution
result_dicts = []  # Build list of dicts
for i in range(len(demo_chunks)):  # Loop through results
    result_dict = {  # Create dict for each result
        "document": demo_chunks[i],  # Chunk text
        "metadata": demo_meta[i]  # Metadata
    }  # End dict
    result_dicts.append(result_dict)  # Add to list

context_with_sources = assemble_context_with_sources(result_dicts)  # Assemble

print("Query:", demo_query)  # Show query
print()  # Blank line
print("Assembled Context (what LLM will see):")  # Label
print("-" * 40)  # Separator
print(context_with_sources)  # Show context
print("-" * 40)  # Separator
print()  # Blank line

# Show the full system prompt with context inserted
full_prompt = GROUNDING_SYSTEM_PROMPT.format(context=context_with_sources)  # Insert context
print("Full System Prompt (first 300 chars):")  # Label
print(full_prompt[:300] + "...")  # Preview
print()  # Blank line

# Test heuristic grounding evaluation
grounded_answer = "According to the policy, digital products are non-refundable after purchase."  # Good answer
hallucinated_answer = "Most companies offer a standard 30-day return window for all products."  # Bad answer

context_text = assemble_context(demo_chunks)  # Plain context for evaluation

is_grounded_good, ratio_good = evaluate_grounding_heuristic(grounded_answer, context_text)  # Evaluate good
is_grounded_bad, ratio_bad = evaluate_grounding_heuristic(hallucinated_answer, context_text)  # Evaluate bad

print("Grounding Evaluation (heuristic):")  # Label
print(f"  ✅ Grounded answer: is_grounded={is_grounded_good}, overlap={ratio_good:.2f}")  # Good result
print(f"     {grounded_answer[:70]}...")  # Show answer
print(f"  ❌ Hallucinated answer: is_grounded={is_grounded_bad}, overlap={ratio_bad:.2f}")  # Bad result
print(f"     {hallucinated_answer[:70]}...")  # Show answer
print()  # Blank line
print("Note: In production, use an LLM-based faithfulness scorer (e.g., RAGAS)")  # Tip
print("for more reliable grounding evaluation than this word-overlap heuristic.")  # Tip

---
## Section 8: Evaluation Metrics

### Retrieval Metrics

**Precision@K** — Of the top-K chunks, what fraction is relevant?
```
Precision@K = (# relevant chunks in top-K) / K

Score 1.0 = every retrieved chunk is relevant (perfect precision)
Score 0.0 = no retrieved chunk is relevant
```

**Recall@K** — Of ALL relevant chunks in the database, what fraction did we retrieve?
```
Recall@K = (# relevant chunks in top-K) / (total # relevant chunks in database)

Score 1.0 = found all relevant chunks (perfect recall)
Score 0.5 = found half of all relevant chunks
```

**MRR (Mean Reciprocal Rank)** — Average rank of the **first** relevant result.
```
If first relevant result is at rank 1 → reciprocal rank = 1/1 = 1.0
If first relevant result is at rank 2 → reciprocal rank = 1/2 = 0.5
If first relevant result is at rank 3 → reciprocal rank = 1/3 = 0.33

MRR = average reciprocal rank across all queries
```

**NDCG@K (Normalized Discounted Cumulative Gain)** — Ranked relevance, penalising
lower-ranked results more.
```
NDCG 1.0 = perfect ranking (all relevant at top)
NDCG 0.7 = good ranking (relevant mostly at top)
NDCG 0.3 = poor ranking (relevant scattered throughout)
```

### Generation Metrics

| Metric | Question It Answers | How Measured |
|---|---|---|
| **Relevance** | Does the answer address the question? | Human or LLM eval |
| **Faithfulness** | Is the answer supported by retrieved context? | Human or RAGAS |
| **Correctness** | Does the answer match ground truth? | F1, BLEU, exact match |

### RAGAS Framework

**RAGAS (Retrieval-Augmented Generation Assessment)** is an open-source framework
that automatically measures:
- `faithfulness` — % of answer claims supported by context
- `answer_relevancy` — % of answer that addresses the question
- `context_relevance` — % of retrieved context that is relevant
- `context_recall` — % of relevant chunks captured in retrieval

**Typical production targets:**
- Faithfulness > 0.80
- Answer Relevancy > 0.75
- Context Recall > 0.70


In [ ]:
# ── Section 8 Demo: Computing retrieval metrics manually ─────────────


# ─────────────────────────────────────────────────────────────────────
# Metric 1: Precision@K
# ─────────────────────────────────────────────────────────────────────
def precision_at_k(retrieved_ids, relevant_ids, k):
    """
    Compute Precision@K.

    Args:
        retrieved_ids (list): IDs of top-K retrieved chunks
        relevant_ids (set): IDs of all truly relevant chunks
        k (int): Number of results to evaluate

    Returns:
        float: Precision@K score (0.0 to 1.0)
    """
    top_k = retrieved_ids[:k]  # Consider only top-K results
    relevant_count = 0  # Count relevant results

    for chunk_id in top_k:  # Loop through top-K
        if chunk_id in relevant_ids:  # Is this chunk relevant?
            relevant_count = relevant_count + 1  # Increment count

    if k == 0:  # Avoid division by zero
        return 0.0  # Return 0 if K is 0

    return relevant_count / k  # Precision = relevant / K


# ─────────────────────────────────────────────────────────────────────
# Metric 2: Recall@K
# ─────────────────────────────────────────────────────────────────────
def recall_at_k(retrieved_ids, relevant_ids, k):
    """
    Compute Recall@K.

    Args:
        retrieved_ids (list): IDs of top-K retrieved chunks
        relevant_ids (set): IDs of all truly relevant chunks
        k (int): Number of results to evaluate

    Returns:
        float: Recall@K score (0.0 to 1.0)
    """
    top_k = retrieved_ids[:k]  # Consider only top-K results
    relevant_count = 0  # Count relevant results found

    for chunk_id in top_k:  # Loop through top-K
        if chunk_id in relevant_ids:  # Is this chunk relevant?
            relevant_count = relevant_count + 1  # Increment count

    total_relevant = len(relevant_ids)  # Total relevant in database

    if total_relevant == 0:  # No relevant chunks exist
        return 0.0  # Recall undefined; return 0

    return relevant_count / total_relevant  # Recall = found / total relevant


# ─────────────────────────────────────────────────────────────────────
# Metric 3: MRR (Mean Reciprocal Rank)
# ─────────────────────────────────────────────────────────────────────
def mean_reciprocal_rank(all_retrieved_ids, all_relevant_ids):
    """
    Compute Mean Reciprocal Rank across multiple queries.

    Args:
        all_retrieved_ids (list of lists): Retrieved IDs for each query
        all_relevant_ids (list of sets): Relevant IDs for each query

    Returns:
        float: MRR score (0.0 to 1.0)
    """
    reciprocal_ranks = []  # Accumulate per-query reciprocal ranks

    for query_idx in range(len(all_retrieved_ids)):  # Loop through each query
        retrieved = all_retrieved_ids[query_idx]  # Retrieved IDs for this query
        relevant = all_relevant_ids[query_idx]  # Relevant IDs for this query
        reciprocal_rank = 0.0  # Default: no relevant result found

        for rank_idx in range(len(retrieved)):  # Loop through retrieved results
            chunk_id = retrieved[rank_idx]  # Current chunk ID
            if chunk_id in relevant:  # Found first relevant result
                reciprocal_rank = 1.0 / (rank_idx + 1)  # Rank is 1-indexed
                break  # Stop at first relevant result

        reciprocal_ranks.append(reciprocal_rank)  # Add to list

    if not reciprocal_ranks:  # No queries
        return 0.0  # Return 0

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)  # Mean of reciprocal ranks
    return mrr  # Return MRR


# ─────────────────────────────────────────────────────────────────────
# Metric 4: NDCG@K
# ─────────────────────────────────────────────────────────────────────
def ndcg_at_k(retrieved_ids, relevant_ids, k):
    """
    Compute NDCG@K (binary relevance: relevant=1, not relevant=0).

    Args:
        retrieved_ids (list): IDs of top-K retrieved chunks
        relevant_ids (set): IDs of all truly relevant chunks
        k (int): Number of results to evaluate

    Returns:
        float: NDCG@K score (0.0 to 1.0)
    """
    import math  # For log computation

    top_k = retrieved_ids[:k]  # Consider only top-K results

    # Compute DCG (Discounted Cumulative Gain)
    dcg = 0.0  # Accumulate DCG
    for i in range(len(top_k)):  # Loop through top-K (0-indexed)
        relevance = 1.0 if top_k[i] in relevant_ids else 0.0  # Binary relevance
        position = i + 1  # Position is 1-indexed
        if position == 1:  # Avoid log(1) = 0 in denominator
            discount = 1.0  # No discount for position 1
        else:  # Apply logarithmic discount
            discount = math.log2(position)  # Discount = log2(position)
        dcg = dcg + (relevance / discount)  # Add to DCG

    # Compute IDCG (Ideal DCG — all relevant results at top)
    num_relevant = min(len(relevant_ids), k)  # How many relevant results can we have?
    idcg = 0.0  # Accumulate IDCG
    for i in range(num_relevant):  # Loop through ideal positions
        position = i + 1  # 1-indexed
        if position == 1:  # No discount for position 1
            discount = 1.0  # No discount
        else:  # Logarithmic discount
            discount = math.log2(position)  # Discount
        idcg = idcg + (1.0 / discount)  # All ideal results are relevant

    if idcg == 0.0:  # No relevant chunks exist
        return 0.0  # NDCG undefined; return 0

    return dcg / idcg  # Normalise DCG by IDCG


# ─────────────────────────────────────────────────────────────────────
# Run metric evaluations on simulated retrieval results
# ─────────────────────────────────────────────────────────────────────
print("=" * 60)  # Separator
print("EVALUATION METRICS DEMO")  # Header
print("=" * 60)  # Separator
print()  # Blank line

# Simulate retrieval for query: "Can I return digital products?"
# Assume truly relevant chunks are refund_digital_1 and refund_physical_1
truly_relevant = {"refund_digital_1", "refund_physical_1"}  # Ground truth relevance

# Retrieve from our collection
eval_query = "Can I return digital products?"  # Evaluation query
eval_query_emb = model.encode(eval_query).tolist()  # Embed query
eval_results = collection.query(  # Retrieve
    query_embeddings=[eval_query_emb],  # Embedded query
    n_results=5  # Get top-5
)  # End query

retrieved_chunk_ids = eval_results["ids"][0]  # Retrieved IDs in rank order

print(f"Query: {eval_query}")  # Show query
print(f"Truly relevant chunks: {truly_relevant}")  # Show ground truth
print(f"Retrieved order: {retrieved_chunk_ids}")  # Show retrieval order
print()  # Blank line

# Compute all metrics
p_at_3 = precision_at_k(retrieved_chunk_ids, truly_relevant, k=3)  # Precision@3
r_at_3 = recall_at_k(retrieved_chunk_ids, truly_relevant, k=3)  # Recall@3
p_at_5 = precision_at_k(retrieved_chunk_ids, truly_relevant, k=5)  # Precision@5
r_at_5 = recall_at_k(retrieved_chunk_ids, truly_relevant, k=5)  # Recall@5
ndcg_3 = ndcg_at_k(retrieved_chunk_ids, truly_relevant, k=3)  # NDCG@3

# MRR across simulated multi-query evaluation
all_retrieved = [  # Retrieved IDs for 3 queries
    retrieved_chunk_ids,  # Query 1 results
    ["shipping_1", "refund_digital_1", "warranty_1"],  # Query 2 results (simulated)
    ["refund_physical_1", "shipping_2", "warranty_1"],  # Query 3 results (simulated)
]  # End all_retrieved
all_relevant = [  # Truly relevant for each query
    truly_relevant,  # Query 1 relevant
    {"refund_digital_1"},  # Query 2 relevant (simulated)
    {"refund_physical_1"},  # Query 3 relevant (simulated)
]  # End all_relevant
mrr_score = mean_reciprocal_rank(all_retrieved, all_relevant)  # Compute MRR

print("Retrieval Metrics:")  # Header
print(f"  Precision@3 : {p_at_3:.3f}  (of top-3, {p_at_3*3:.0f}/3 are relevant)")  # Precision@3
print(f"  Precision@5 : {p_at_5:.3f}  (of top-5, {p_at_5*5:.0f}/5 are relevant)")  # Precision@5
print(f"  Recall@3    : {r_at_3:.3f}  (found {r_at_3*len(truly_relevant):.0f}/{len(truly_relevant)} relevant chunks)")  # Recall@3
print(f"  Recall@5    : {r_at_5:.3f}  (found {r_at_5*len(truly_relevant):.0f}/{len(truly_relevant)} relevant chunks)")  # Recall@5
print(f"  NDCG@3      : {ndcg_3:.3f}  (1.0 = perfect ranking)")  # NDCG@3
print(f"  MRR         : {mrr_score:.3f}  (avg rank of first relevant = {1/mrr_score:.1f})")  # MRR
print()  # Blank line
print("Interpretation:")  # Header
print("  Precision tells us: are retrieved chunks mostly relevant?")  # Interpretation
print("  Recall tells us: did we find all the relevant chunks?")  # Interpretation
print("  MRR tells us: is the first relevant result near the top?")  # Interpretation
print("  NDCG tells us: are relevant results ranked higher than irrelevant ones?")  # Interpretation

---
## Section 9: Common Failure Modes & Fixes

Understanding what goes wrong — and why — is as important as building the happy path.

---

### Failure 1: Chunking Mismatch

**Symptom:** Query asks about a topic that spans multiple chunks; no single chunk has the full answer.

**Root cause:** Document chunked by paragraphs, but the answer requires reading multiple sections together.

**Fix:** Use overlapping chunks or semantic chunking to capture cross-boundary context.

---

### Failure 2: Vocabulary Mismatch

**Symptom:** Query uses "late fee" but document says "grace period charge" — low similarity, not retrieved.

**Root cause:** Embedding model trained on general text doesn't know your domain synonyms.

**Fixes:**
- Query expansion (generate synonym variants)
- Domain-specific embedding model (fine-tuned on your domain)
- Hybrid retrieval (BM25 catches exact-term variants)

---

### Failure 3: Noise in Retrieved Context

**Symptom:** Retrieved K=3 chunks, but 1–2 are marginally relevant. LLM gets confused by noise.

**Root cause:** Dense retrieval doesn't distinguish "somewhat related" from "highly relevant."

**Fixes:**
- Re-ranking (cross-encoder filters irrelevant candidates)
- Raise similarity threshold (filter chunks below 0.75)
- Reduce K (fewer chunks = less noise)

---

### Failure 4: Hallucination Despite Grounding

**Symptom:** LLM generates an answer using training knowledge even though context is provided.

**Root cause:** LLM treats grounding prompt as a soft suggestion, not a hard constraint.

**Fixes:**
- Stricter system prompt (use stronger negative prompting)
- Context compression (summarise context before passing — focuses LLM)
- Use a smaller model (less world knowledge = less temptation to hallucinate)

---

### Failure 5: No Grounding Enforcement

**Symptom:** No grounding system prompt → LLM mixes training knowledge + context freely.

**Root cause:** Without explicit instruction, LLMs default to treating context as supplementary, not exclusive.

**Fix:** Always include explicit grounding system prompt. Audit generated answers regularly.

---

### Failure 6: Context Length Overflow

**Symptom:** Too many chunks passed to LLM → LLM "loses track" of relevant information.

**Root cause:** Attention mechanism diluted by too many tokens.

**Fix:** Use re-ranking to keep only top-2 or top-3 chunks. Aim for < 2,000 tokens of context.


In [ ]:
# ── Section 9 Demo: Diagnosing failure modes ─────────────────────────

print("=" * 60)  # Separator
print("COMMON FAILURE MODES & DIAGNOSIS")  # Header
print("=" * 60)  # Separator
print()  # Blank line

# ─────────────────────────────────────────────────────────────────────
# Failure 1: Vocabulary Mismatch
# ─────────────────────────────────────────────────────────────────────
print("Failure 1: Vocabulary Mismatch")  # Label
print("-" * 40)  # Separator

# Add a chunk with domain-specific terminology not in original collection
try:  # May fail if ID already exists
    collection.add(  # Add new chunk
        ids=["billing_1"],  # Unique ID
        documents=["Accounts overdue by 30 days incur a grace period charge of $25."],  # Domain jargon
        embeddings=[model.encode("Accounts overdue by 30 days incur a grace period charge of $25.").tolist()],  # Embedding
        metadatas=[{"source": "billing_policy.pdf", "chunk_index": 0, "category": "billing",  # Metadata
                    "date_added": "2024-06-01", "language": "en"}]  # More metadata
    )  # End add
except Exception:  # Already added
    pass  # Ignore duplicate

# Query using different terminology
mismatch_query = "What is the late fee on unpaid accounts?"  # Uses "late fee"
mismatch_query_emb = model.encode(mismatch_query).tolist()  # Embed query
mismatch_results = collection.query(  # Retrieve
    query_embeddings=[mismatch_query_emb],  # Embedded query
    n_results=3  # Top-3
)  # End query

print(f"Query: {mismatch_query}")  # Show query
print("Retrieved chunks:")  # Label
for i in range(len(mismatch_results["documents"][0])):  # Loop through results
    doc = mismatch_results["documents"][0][i]  # Chunk text
    dist = mismatch_results["distances"][0][i]  # Distance
    sim = 1 / (1 + dist)  # Convert to similarity
    print(f"  Rank {i + 1} | sim={sim:.4f}: {doc[:70]}")  # Show result
print()  # Blank line
print("Diagnosis: The billing chunk (grace period charge) may rank low")  # Analysis
print("because 'late fee' and 'grace period charge' have low embedding similarity.")  # Analysis
print()  # Blank line

# Fix: Query expansion
expanded_for_mismatch = [  # Multiple phrasings for "late fee"
    "What is the late fee on unpaid accounts?",  # Original
    "What is the grace period charge?",  # Synonym
    "What happens when my account is overdue?",  # Alternative
    "What charges apply for overdue accounts?",  # Another alternative
]  # End expanded queries

expansion_vote = {}  # Vote map: chunk_text → vote count
for q in expanded_for_mismatch:  # Retrieve for each variant
    q_emb = model.encode(q).tolist()  # Embed variant
    q_res = collection.query(query_embeddings=[q_emb], n_results=2)  # Top-2
    for doc in q_res["documents"][0]:  # Loop through results
        if doc not in expansion_vote:  # New chunk
            expansion_vote[doc] = 0  # Initialise
        expansion_vote[doc] = expansion_vote[doc] + 1  # Vote

sorted_expansion = sorted(expansion_vote.items(), key=lambda x: x[1], reverse=True)  # Sort
print("Fix: Query Expansion results:")  # Label
for rank, (text, votes) in enumerate(sorted_expansion[:3]):  # Top-3
    print(f"  Rank {rank + 1} | Votes: {votes}: {text[:70]}")  # Show
print()  # Blank line

# ─────────────────────────────────────────────────────────────────────
# Failure 2: Grounding Failure Simulation
# ─────────────────────────────────────────────────────────────────────
print("Failure 2: Grounding Failure Simulation")  # Label
print("-" * 40)  # Separator

# Simulate an "I don't know" scenario — query not in collection
out_of_scope_query = "What is the employee vacation policy?"  # Not in our documents
out_of_scope_emb = model.encode(out_of_scope_query).tolist()  # Embed
oos_results = collection.query(  # Retrieve — will find "closest" but irrelevant
    query_embeddings=[out_of_scope_emb],  # Embedded query
    n_results=3  # Top-3
)  # End query

oos_chunks = oos_results["documents"][0]  # Retrieved chunks
oos_dists = oos_results["distances"][0]  # Distances

print(f"Out-of-scope query: {out_of_scope_query}")  # Show query
print("Retrieved (closest, but irrelevant):")  # Label
for i in range(len(oos_chunks)):  # Loop through results
    sim = 1 / (1 + oos_dists[i])  # Convert to similarity
    print(f"  Rank {i + 1} | sim={sim:.4f}: {oos_chunks[i][:60]}")  # Show
print()  # Blank line

# Show the max similarity — if it's low, nothing relevant was retrieved
max_sim = max([1 / (1 + d) for d in oos_dists])  # Highest similarity
print(f"Max similarity: {max_sim:.4f}")  # Show max

if max_sim < 0.6:  # Low similarity threshold
    print("Adaptive retrieval: max similarity below 0.6 threshold.")  # Diagnosis
    print("A well-grounded system should say: \"I don't have enough information.\"")  # Expected
else:  # High similarity
    print("Similarity above threshold — context passed to LLM.")  # Continue normally
print()  # Blank line

# ─────────────────────────────────────────────────────────────────────
# Failure 3: Context Length Check
# ─────────────────────────────────────────────────────────────────────
print("Failure 3: Context Length Overflow Check")  # Label
print("-" * 40)  # Separator

k_values = [1, 3, 5, 10]  # Different K values to test

for k_val in k_values:  # Loop through K values
    test_results = collection.query(  # Retrieve K chunks
        query_embeddings=[mismatch_query_emb],  # Reuse embedded query
        n_results=min(k_val, collection.count())  # Don't exceed collection size
    )  # End query

    retrieved_texts = test_results["documents"][0]  # Retrieved texts
    total_chars = sum(len(t) for t in retrieved_texts)  # Total characters
    approx_tokens = total_chars // 4  # Approximate token count

    warning = ""  # Default no warning
    if approx_tokens > 2000:  # Exceeds recommended context
        warning = " ⚠️ May cause LLM focus loss!"  # Warning

    print(f"  K={k_val}: ~{approx_tokens} tokens of context{warning}")  # Show

print()  # Blank line
print("Recommendation: K=3 with re-ranking is the safest default.")  # Summary
print("Aim for < 2,000 tokens of context to maintain LLM focus.")  # Tip

---
## Section 10: End-to-End RAG System

### Complete Pipeline Integration

This section integrates all techniques into a single `RAGSystem` class:

1. **Ingest:** chunk → embed → store (with metadata)
2. **Retrieve:** embed query → similarity search → optional re-ranking
3. **Generate:** assemble context → grounding prompt → LLM call

> ⚠️ **This section requires `ANTHROPIC_API_KEY`.**
> Set it via secrets


In [ ]:
# ── Section 10: Full RAG System (requires ANTHROPIC_API_KEY) ─────────

import os  # For reading environment variables
import anthropic  # Claude API client
from google.colab import userdata

class RAGSystem:
    """
    Complete RAG pipeline: chunk → embed → store → retrieve → generate.
    Integrates all techniques from Sections 1-9.
    """

    def __init__(self, collection_name="rag_system"):
        """
        Initialise the RAG system.

        Args:
            collection_name (str): Name of the ChromaDB collection
        """
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # Embedding model
        self.chroma_client = chromadb.Client()  # In-memory vector store

        # Create or get collection
        try:  # Try creating collection
            self.collection = self.chroma_client.create_collection(  # Create new
                name=collection_name  # Collection name
            )  # End create
        except Exception:  # Collection may already exist
            self.collection = self.chroma_client.get_collection(  # Get existing
                name=collection_name  # Collection name
            )  # End get
        api_key = userdata.get('MY_API_KEY')
        self.anthropic_client = anthropic.Anthropic(api_key=api_key)  # Claude API client

        # Grounding system prompt with context placeholder
        self.system_prompt_template = GROUNDING_SYSTEM_PROMPT  # Use constant defined in Section 7

    def ingest(self, documents, metadata_list):
        """
        Ingest documents: chunk → embed → store in vector database.

        Args:
            documents (list): List of raw document strings
            metadata_list (list): List of metadata dicts, one per document
        """
        total_chunks = 0  # Track total chunks added
        chunk_id = 0  # Global chunk counter

        for doc_idx in range(len(documents)):  # Loop through each document
            doc_text = documents[doc_idx]  # Raw document text
            doc_meta = metadata_list[doc_idx]  # Document metadata

            # Stage 1: Chunk using sentence-aware strategy
            chunks = sentence_aware_chunking(doc_text, chunk_size_tokens=200)  # Chunk

            for chunk_idx in range(len(chunks)):  # Loop through chunks
                chunk = chunks[chunk_idx]  # Current chunk text

                # Stage 2: Embed the chunk
                embedding = self.embed_model.encode(chunk)  # Compute embedding
                embedding_list = embedding.tolist()  # Convert to Python list

                # Build chunk metadata (inherit doc metadata + add chunk info)
                chunk_meta = {}  # Start fresh
                for key in doc_meta:  # Copy all doc metadata
                    chunk_meta[key] = doc_meta[key]  # Copy key
                chunk_meta["chunk_index"] = chunk_idx  # Add chunk position
                chunk_meta["doc_index"] = doc_idx  # Add doc position

                chunk_id_str = f"rag_chunk_{chunk_id}"  # Unique ID

                # Stage 3: Store in ChromaDB
                self.collection.add(  # Add chunk
                    ids=[chunk_id_str],  # Unique ID
                    documents=[chunk],  # Chunk text
                    embeddings=[embedding_list],  # Pre-computed embedding
                    metadatas=[chunk_meta]  # Metadata
                )  # End add

                chunk_id = chunk_id + 1  # Increment global counter
                total_chunks = total_chunks + 1  # Increment total

        print(f"✅ Ingested {len(documents)} documents → {total_chunks} chunks")  # Summary

    def retrieve(self, query, k=3):
        """
        Retrieve top-K most relevant chunks for a query.

        Args:
            query (str): User question
            k (int): Number of chunks to retrieve

        Returns:
            tuple: (chunks list, metadata list)
        """
        # Embed query using same model as index time
        query_emb = self.embed_model.encode(query).tolist()  # Embed query

        # Retrieve from ChromaDB
        results = self.collection.query(  # Similarity search
            query_embeddings=[query_emb],  # Query embedding
            n_results=min(k, self.collection.count())  # Don't exceed collection size
        )  # End query

        chunks = results["documents"][0]  # Retrieved chunk texts
        metadata = results["metadatas"][0]  # Retrieved metadata

        return chunks, metadata  # Return results

    def generate(self, query):
        """
        Full RAG pipeline: retrieve + generate grounded answer.

        Args:
            query (str): User question

        Returns:
            dict: answer, retrieved_chunks, metadata
        """
        # Stage 4: Retrieve
        chunks, metadata = self.retrieve(query, k=3)  # Get top-3 chunks

        # Assemble context with source attribution
        result_dicts = []  # Build list of dicts
        for i in range(len(chunks)):  # Loop through chunks
            result_dicts.append({"document": chunks[i], "metadata": metadata[i]})  # Dict
        context = assemble_context_with_sources(result_dicts)  # Assemble context

        # Stage 5: Generate grounded answer
        system_prompt = self.system_prompt_template.format(context=context)  # Insert context

        response = self.anthropic_client.messages.create(  # Call Claude API
            model="claude-haiku-4-5-20251001",  # Use Claude Sonnet 4
            max_tokens=1000,  # Max response tokens
            system=system_prompt,  # Grounding prompt with context
            messages=[  # Conversation messages
                {"role": "user", "content": query}  # User question
            ]  # End messages
        )  # End API call

        answer = response.content[0].text  # Extract text from response

        return {  # Return full result dict
            "answer": answer,  # Generated answer
            "retrieved_chunks": chunks,  # Chunks used as context
            "metadata": metadata  # Metadata for retrieved chunks
        }  # End return dict


# ─────────────────────────────────────────────────────────────────────
# Demo: Run end-to-end RAG on sample documents
# ─────────────────────────────────────────────────────────────────────

    # Define sample documents
sample_docs = [  # List of documents to ingest
        """Refund Policy

Digital products are non-refundable after purchase and download.
Physical goods can be returned within 30 days of purchase in original packaging.
Shipping costs are non-refundable.
Return shipping is free for defective items only.
Proof of purchase is required for all returns.""",  # Refund policy doc

        """Warranty Policy

Electronics have a 1-year manufacturer warranty covering defects.
Batteries are covered for 6 months.
Physical damage and water damage void the warranty.
Extended warranties are available for purchase at checkout.""",  # Warranty policy doc

        """Shipping Policy

Standard shipping takes 5-7 business days.
Express shipping takes 1-2 business days at extra cost.
International shipping is available to 40 countries.
Orders over $50 qualify for free standard shipping."""  # Shipping policy doc
    ]  # End sample docs

sample_metadata = [  # Metadata for each document
        {"source": "refund_policy.txt", "category": "refunds",  # Doc 1 metadata
         "date_added": "2024-06-01", "language": "en"},  # More metadata
        {"source": "warranty_policy.txt", "category": "warranty",  # Doc 2 metadata
         "date_added": "2024-06-01", "language": "en"},  # More metadata
        {"source": "shipping_policy.txt", "category": "shipping",  # Doc 3 metadata
         "date_added": "2024-06-01", "language": "en"}  # More metadata
    ]  # End sample metadata

    # Initialise and ingest
rag = RAGSystem(collection_name="demo_rag")  # Create RAG system
rag.ingest(sample_docs, sample_metadata)  # Ingest documents
print()  # Blank line

# Test queries
test_queries = [  # Questions to ask
        "Can I return a digital product for a refund?",  # Should answer: non-refundable
        "How long is the warranty on batteries?",  # Should answer: 6 months
        "How long does standard shipping take?",  # Should answer: 5-7 business days
        "What is your employee vacation policy?",  # Out of scope: I don't know
    ]  # End test queries

print("=" * 60)  # Separator
print("END-TO-END RAG DEMO")  # Header
print("=" * 60)  # Separator

for i in range(len(test_queries)):  # Loop through test queries
        q = test_queries[i]  # Current query
        print(f"\nQ{i + 1}: {q}")  # Print query
        print("-" * 40)  # Separator

        result = rag.generate(q)  # Run full RAG pipeline

        print("Retrieved chunks:")  # Label
        for j in range(len(result["retrieved_chunks"])):  # Loop through chunks
            chunk_text = result["retrieved_chunks"][j]  # Chunk text
            chunk_src = result["metadata"][j]["source"]  # Source
            print(f"  [{chunk_src}] {chunk_text[:60]}...")  # Preview

        print(f"\nAnswer: {result['answer']}")  # Print answer
        print()  # Blank line